# User-defined Wake Models

Beginning in v5, FLORIS supports user-defined wake models that can be passed directly into FLORIS using `fmodel.set_wake_model`. A user-defined wake model may be a dynamic or static class, but will usually be dynamic to allow model parameters to be set as attributes. It must conform to the `attrs` package for declaring attributes (in particular, wake model parameters). Additionally all user-defined operation models should inherit from the abstract parent class `BaseWakeModel`, available in FLORIS.

All operation models must implement the following "fundamental" methods:
- `turbine_solve`: computes the flow solution at all turbine locations, part of the main FLORIS `run` procedure.
- `point_solve`: computes the flow solution at arbitrary, user-provided points in the flow, or for cut planes for visualization purposes.

Wake models may then implement additional methods as needed.

The following arguments are passed to either `turbine_solve` or `point_solve` at runtime:

| Argument | Data type | Description |
|----------|-----------|----------|
| `farm` | `floris.core.Farm` | text |
| `flow_field` | `floris.core.FlowField` | The flow field object, which contains the flow solution and other flow-related quantities. |
| `grid` | `floris.core.TurbineGrid` or `floris.core.FlowFieldPlanarGrid` or `floris.core.PointsGrid` | The grid object corresponding to the type of solve being performed. For `turbine_solve`, this will be a `TurbineGrid`. For `point_solve`, this will be either a `FlowFieldPlanarGrid` or `PointsGrid`, depending on the type of points being solved for (visualization-type solves or individual point solves, respectively). |

The `turbine_solve` and `point_solve` methods do not return any values, but instead update the `flow_field` argument in-place.

### Static example

We begin with a very simple example that will produce a "straight" wake behind each turbine, whose velocity deficit (as a fraction of the free stream velocity) is constant and user-definable. This is not a good wake model (and doesn't adhere to momentum conservation)! We're just using it as a basic example to demonstrate the functionality.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from attrs import define, field
from floris.type_dec import floris_float_type, NDArrayFloat
from floris.core.wake_model import BaseWakeModel
from floris.flow_visualization import visualize_cut_plane

from floris.core import (
    BaseModel,
    Farm,
    FlowField,
    FlowFieldPlanarGrid,
    PointsGrid,
    TurbineGrid,
)

@define
class StraightWake(BaseWakeModel):
    """
    A simple wake model that produces a straight wake behind each turbine.
    """

    # Using attrs, we can define model parameters as class attributes.
    velocity_deficit: float = field(default=0.2)
    wake_width: float = field(default=100.0)

    # Define a method for determining whether a test point is within the wake of at least one
    # turbine
    def _is_in_wake(self, grid, turbine_i_x, turbine_i_y, turbine_i_z):

        # Declare all True to start
        in_wake_i = np.full(grid.x_sorted.shape, True)

        # Check if downstream of any turbine
        in_wake_i &= (grid.x_sorted > turbine_i_x.mean(axis=(2,3), keepdims=True))

        # Check if within wake width of any turbine
        in_wake_i &= (
            np.abs(grid.y_sorted - turbine_i_y.mean(axis=(2,3), keepdims=True))
            < self.wake_width / 2
        )
        in_wake_i &= (
            np.abs(grid.z_sorted - turbine_i_z.mean(axis=(2,3), keepdims=True))
            < self.wake_width / 2
        )

        # Return resulting boolean array
        return in_wake_i

    # Define the main turbine_solve method for solving at turbine locations
    def turbine_solve(
        self,
        farm: Farm,
        flow_field: FlowField,
        grid: TurbineGrid,
    ) -> None:

        # Initialize an array to keep track of whether each point is in the wake of any turbine
        in_wake = np.full(grid.x_sorted.shape, False)

        for i in range(grid.n_turbines):

            # Check if the points are in the wake of turbine i
            in_wake_i = self._is_in_wake(
                grid,
                grid.x_sorted[:, i:i+1, :, :],
                grid.y_sorted[:, i:i+1, :, :],
                grid.z_sorted[:, i:i+1, :, :]
            )

            # Update the overall in_wake array to include the wake of turbine i
            in_wake |= in_wake_i

        # Apply velocity deficits
        flow_field.u_sorted = flow_field.u_initial_sorted * (1 - self.velocity_deficit * in_wake)

        self.evaluate_turbine_power(grid, farm, flow_field)
        self.evaluate_turbine_thrust_coefficient(grid, farm, flow_field)
        print("turbine_solve completed with StraightWake model!")

        return None

    # Define the secondary point_solve method for solving at arbitrary points in the flow
    def point_solve(
        self,
        farm: Farm,
        flow_field: FlowField,
        grid: FlowFieldPlanarGrid | PointsGrid,
    ) -> None:
        # Use parent class method to access the turbine grid
        turbine_grid = self.generate_turbine_grid_objects(farm, flow_field)[2]

        # Initialize an array to keep track of whether each point is in the wake of any turbine
        in_wake = np.full(grid.x_sorted.shape, False)

        for i in range(turbine_grid.n_turbines):

            # Check if the turbine is in the wake of any other turbine
            in_wake_i = self._is_in_wake(
                grid,
                turbine_grid.x_sorted[:, i:i+1, :, :],
                turbine_grid.y_sorted[:, i:i+1, :, :],
                turbine_grid.z_sorted[:, i:i+1, :, :]
            )

            # Update the overall in_wake array to include the wake of turbine i
            in_wake |= in_wake_i

        # Apply velocity deficits
        flow_field.u_sorted = flow_field.u_initial_sorted * (1 - self.velocity_deficit * in_wake)
        print("point_solve completed with StraightWake model!")

        return None

Let's now use this straight wake model in FLORIS.

In [ ]:
from floris import FlorisModel, TimeSeries

fmodel = FlorisModel("defaults")
time_series = TimeSeries(
    wind_directions=np.array([270.0, 270.0, 280.0]),
    wind_speeds=np.array([8.0, 10.0, 12.0]),
    turbulence_intensities=np.array([0.06, 0.06, 0.06]),
)
fmodel.set(
    layout_x = [0.0, 500.0],
    layout_y = [0.0, 0.0],
    wind_data=time_series,
)
fmodel.set_wake_model(StraightWake(velocity_deficit=0.2, wake_width=100.0))

fmodel.run()

print("Powers [W]:\n", fmodel.get_turbine_powers(), "\n")
print("Thrust coefficients [-]:\n", fmodel.get_turbine_thrust_coefficients(), "\n")

## Visualization example

Now, we will perform a flow visualization, which uses the `point_solve` method.

In [ ]:
# Rotate flow to visualize separate wakes
fmodel.set(wind_speeds=[8.0], wind_directions=[280.0], turbulence_intensities=[0.06])

horizontal_plane = fmodel.calculate_horizontal_plane(
    x_resolution=200,
    y_resolution=100,
    height=90.0,
)

fig, ax = plt.subplots()
visualize_cut_plane(
    horizontal_plane,
    ax=ax,
    label_contours=False,
    title="Horizontal Flow with Turbine Rotors and labels",
)

## Prepackaged wake models

Naturally, prepackaged wake models can also be used in this way. Let's take a look at using the `Gauss` wake model from FLORIS, either as one of the preset defaults or by passing the class in directly.

In [ ]:
from floris.core.wake_model import Gauss

fmodel.set_wake_model(Gauss()) # Use Gauss defaults
horizontal_plane = fmodel.calculate_horizontal_plane(
    x_resolution=200,
    y_resolution=100,
    height=90.0,
)

fig, ax = plt.subplots()
visualize_cut_plane(
    horizontal_plane,
    ax=ax,
    label_contours=False,
    title="Horizontal Flow with Turbine Rotors and labels",
)